# Figs. A2 and A8 — Zak phases of the driven diamond chain

Zak phases $\mathcal{Z}_j = i \oint dk\, \langle u_j(k)|\partial_k|u_j(k)\rangle$ of the three bands ($j = 1, 2, 3$ by increasing
(quasi)energy) for antisymmetric driving ($\zeta = -1$), $\omega = 8$, $J = 1$, $\phi = \pi$, as a function of the amplitude $A$:

1. **Exact Floquet bands**: stroboscopic Floquet modes $|u_j(k)\rangle$ (eigenvectors of the exact one-period operator $\hat U_{\rm t}(T)$), as stated in MT Secs. II and IV (Fig. A2).
2. **Second-order effective model**: eigenvectors of $\hat H_{\rm eff}(k)$, Eq. (h_eff_total), with the corrected $\Gamma$, as stated in Appendix A8 (Fig. A8).
3. **Dependence on the time $t_0$** at which the Floquet modes are taken.

Functions: `Source/zak_phase.py` (Wilson loop, exact Floquet Bloch modes) and `effective_model.Hk_effective`.
Rewritten from `Codes/Python/topological_invariant.ipynb` (which used QuTiP's `FloquetBasis` and the old $\Gamma$).

The Wilson loop runs over a periodic $k$ grid and closes with the link $k_{N-1} \to k_0$, so it is gauge invariant.
The basis is $(C, U, D)$ of `diamond_chain.Hk_static`.

Output: `Data/FigA02_A08_zak_phase.npz`. Runtime: ~1 min.

In [1]:
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parents[1]                 # .../paper_reproduction
DATA = ROOT / 'Data'

from zak_phase import periodic_kgrid, zak_phases, floquet_bloch_modes, effective_bloch_states
from data_io import save_data

In [2]:
J = 1.0
phi = np.pi
zeta = -1
w = 8
nk = 400
kvals = periodic_kgrid(nk)

Avals = np.linspace(0, 10, 101)              # includes A = 9.6 (Atilde = 1.2) used in the manuscript
A_t0 = 9.6                                   # amplitude of the t0 scan
t0_over_T = np.linspace(0, 1, 41)

## 1. Exact Floquet bands, stroboscopic modes ($t_0 = 0$)

The minimal gaps between neighboring Floquet bands are also stored: the Zak phase of a band is only defined while it is isolated.

In [3]:
Z_exact, gap_min = [], []
for A in Avals:
    quasienergies, modes = floquet_bloch_modes(kvals, phi, A, w, zeta, J)
    Z_exact.append(zak_phases(modes))
    gap_min.append(np.diff(quasienergies, axis=1).min())
Z_exact, gap_min = np.array(Z_exact), np.array(gap_min)

print(f'smallest gap between Floquet bands: {gap_min.min():.3f} J')
print('distinct |Z_j|/pi over all A:', np.unique(np.round(np.abs(Z_exact) / np.pi, 6), axis=0))

smallest gap between Floquet bands: 0.824 J
distinct |Z_j|/pi over all A: [[0.499992 1.       0.499992]]


## 2. Second-order effective model

$\Gamma$ (and therefore $J_1$, $J_2$) has poles at integer $\tilde A = A/\omega$, so $A = 8$ is excluded; $A = 0$ is replaced by $A = 10^{-4}$.

In [4]:
Atilde_vals = np.where(Avals == 0, 1e-4, Avals) / w
valid = ~np.isclose(Atilde_vals, np.round(Atilde_vals))

Z_effective = np.full((len(Avals), 3), np.nan)
for i in np.where(valid)[0]:
    _, states = effective_bloch_states(kvals, Atilde_vals[i], w, J)
    Z_effective[i] = zak_phases(states)

print('distinct |Z_j|/pi over all A:', np.unique(np.round(np.abs(Z_effective[valid]) / np.pi, 6), axis=0))
print('max | |Z_exact| - |Z_effective| | / pi:', np.nanmax(np.abs(np.abs(Z_exact) - np.abs(Z_effective))) / np.pi)

distinct |Z_j|/pi over all A: [[0.499992 1.       0.499992]]
max | |Z_exact| - |Z_effective| | / pi: 4.499896966407756e-09


## 3. Dependence on $t_0$

Floquet modes $|u_j(k, t_0)\rangle = \hat U(t_0, 0)\,|u_j(k, 0)\rangle$ at $A = 9.6$. The drive is symmetric about $t_0 = 0$ and $t_0 = T/2$ only.

In [5]:
T = 2 * np.pi / w
Z_t0 = np.array([zak_phases(floquet_bloch_modes(kvals, phi, A_t0, w, zeta, J, t0=f * T)[1]) for f in t0_over_T])

for f, Z in zip(t0_over_T[::5], Z_t0[::5]):
    print(f't0/T = {f:.3f}:  Z/pi = {np.round(Z / np.pi, 4)}')

t0/T = 0.000:  Z/pi = [-0.5 -1.  -0.5]
t0/T = 0.125:  Z/pi = [-0.5091 -0.9817 -0.5091]
t0/T = 0.250:  Z/pi = [-0.5196 -0.9607 -0.5196]
t0/T = 0.375:  Z/pi = [-0.5091 -0.9817 -0.5091]
t0/T = 0.500:  Z/pi = [-0.5 -1.  -0.5]
t0/T = 0.625:  Z/pi = [-0.5091 -0.9817 -0.5091]
t0/T = 0.750:  Z/pi = [-0.5196 -0.9607 -0.5196]
t0/T = 0.875:  Z/pi = [-0.5091 -0.9817 -0.5091]
t0/T = 1.000:  Z/pi = [-0.5 -1.  -0.5]


In [6]:
save_data(DATA / 'FigA02_A08_zak_phase.npz',
          parameters={'J': J, 'phi': phi, 'zeta': zeta, 'w': w, 'nk': nk, 'A_t0': A_t0},
          Avals=Avals, Z_exact=Z_exact, Z_effective=Z_effective, gap_min=gap_min, t0_over_T=t0_over_T, Z_t0=Z_t0)

Saved Data/FigA02_A08_zak_phase.npz: Avals, Z_exact, Z_effective, gap_min, t0_over_T, Z_t0
